# Dynamic Topic Modeling for Cluster 6: Work, Jobs & Workplace Life

This notebook uses BERTopic to explore potential topics within **Cluster 6** (Work, jobs, workplace life & worker communities) from the subreddit clustering analysis.

**Pipeline:**
1. Load data and filter to cluster 6 subreddits
2. Preprocess text (handle noise words via CountVectorizer stopwords)
3. Fit BERTopic with KeyBERTInspired representation (reduces noise words)
4. Run dynamic topic modeling (topics over time)
5. Visualize results

## 1. Setup & Imports

In [2]:
import os
import pandas as pd
import numpy as np
from datetime import datetime

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN

import warnings
warnings.filterwarnings('ignore')

## 2. Load Data & Filter Cluster 6

In [3]:
# ===== CONFIGURATION =====
# Change this path to your full dataset when ready
DATA_PATH = "reddit_cleaned_01_13.csv"
CLUSTER_PATH = "subreddit_cluster_summary_k20.csv"
TARGET_CLUSTER = 6          # Which cluster to explore (change to 0-7 for other clusters)
TEXT_COLUMN = "merged_text"  # Column with text to model ("merged_text", "text", or "comments_only")
TIME_COLUMN = "created_utc"  # Column with timestamps

# Output directory for HTML figures and CSV results
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
# =========================

In [4]:
# Load cluster summary and extract cluster 6 subreddits
cluster_df = pd.read_csv(CLUSTER_PATH)
c6 = cluster_df[cluster_df["cluster"] == TARGET_CLUSTER]

print(f"Cluster {TARGET_CLUSTER}: {c6['Name by LLM'].values[0]}")
print(f"Top words: {c6['top_words'].values[0]}")
print(f"Expected subreddits: {c6['n_subreddits'].values[0]}, expected rows: {c6['n_rows'].values[0]}")

# Parse subreddits (semicolon-separated)
cluster6_subreddits = [s.strip() for s in c6["subreddits"].values[0].split(";")]
print(f"\nSubreddits in cluster 6 ({len(cluster6_subreddits)}):")
print(cluster6_subreddits)

Cluster 6: Work, jobs, workplace life & worker communities
Top words: jobs; work; employees; managers; workers; entrepreneur; writers; workplace; construction; ideas
Expected subreddits: 91, expected rows: 431

Subreddits in cluster 6 (91):
['AirBnB', 'AustralianTeachers', 'BestBuyWorkers', 'Business_Ideas', 'CAStateWorkers', 'CallCenterWorkers', 'CanadaJobs', 'Career_Advice', 'Careerio', 'Catswithjobs', 'ConstructionManagers', 'ConstructionTech', 'CorporateFacepalm', 'DataScienceJobs', 'EarnMoneyHub', 'EngineeringManagers', 'EngineeringStudents', 'EnterpriseArchitect', 'Entrepreneur', 'EntrepreneurRideAlong', 'Entrepreneurs', 'FedEmployees', 'HireaWriter', 'JobsAddaa', 'Layoffs', 'LinkedInLunatics', 'Lowes', 'MachineLearningJobs', 'McKinsey_BCG_Bain', 'MedicalWriters', 'Nanny', 'PhDStress', 'PythonJobs', 'RecruitmentAgencies', 'RemoteJobs', 'Resume', 'Salary', 'Startup_Ideas', 'ToxicWorkplace', 'UKJobs', 'VHA_Human_Resources', 'VancouverJobs', 'WFHJobs', 'WebDeveloperJobs', 'WorkAdvic

In [5]:
# Load the full dataset
df_all = pd.read_csv(DATA_PATH)
print(f"Total rows in dataset: {len(df_all)}")

# Filter to cluster 6 subreddits only
df = df_all[df_all["subreddit"].isin(cluster6_subreddits)].copy()
print(f"Rows matching cluster 6 subreddits: {len(df)}")

if len(df) < 20:
    print(f"\n⚠️  WARNING: Only {len(df)} documents found for cluster 6.")
    print("   BERTopic needs more documents for meaningful topic discovery.")
    print("   Please update DATA_PATH to your full dataset CSV.")
    print("   Continuing with available data for demonstration...")

print(f"\nSubreddits found: {df['subreddit'].unique().tolist()}")
print(f"Rows with {TEXT_COLUMN}: {df[TEXT_COLUMN].notna().sum()}")

Total rows in dataset: 7197
Rows matching cluster 6 subreddits: 458

Subreddits found: ['careerguidance', 'jobs', 'sysadmin', 'csMajors', 'coworkerstories', 'antiwork', 'WorkReform', 'careeradvice', 'barista', 'UKJobs', 'overemployed', 'recruitinghell', 'CorporateFacepalm', 'recruiting', 'Salary', 'WFHJobs', 'jobhunting', 'Layoffs', 'managers', 'talesfromthejob', 'workplace_bullying', 'smallbusiness', 'receptionists', 'workfromhome', 'consulting', 'EngineeringStudents', 'Lowes', 'CallCenterWorkers', 'humanresources', 'LinkedInLunatics', 'Entrepreneur', 'WebDeveloperJobs', 'hiring', 'ConstructionTech', 'EngineeringManagers', 'VHA_Human_Resources', 'RemoteJobs', 'work', 'ConstructionManagers', 'Career_Advice', 'employeesOfOracle', 'jobsearchhacks', 'womenEngineers', 'linkedin', 'torontoJobs', 'DataScienceJobs', 'Careerio', 'BestBuyWorkers', 'freelanceWriters', 'JobsAddaa', 'WorkAdvice', 'devopsjobs', 'PythonJobs', 'MachineLearningJobs', 'artbusiness', 'MedicalWriters', 'amazonemployees',

In [6]:
# Drop rows with missing text and prepare documents
df = df.dropna(subset=[TEXT_COLUMN]).reset_index(drop=True)

# Parse timestamps and create quarterly bins
df[TIME_COLUMN] = pd.to_datetime(df[TIME_COLUMN])
df["quarter"] = df[TIME_COLUMN].dt.to_period("Q").dt.start_time  # e.g. 2025-01-01 for Q1

docs = df[TEXT_COLUMN].tolist()
timestamps = df["quarter"].tolist()  # quarterly timestamps for dynamic topic modeling

print(f"Documents ready: {len(docs)}")
print(f"Time range: {df[TIME_COLUMN].min()} to {df[TIME_COLUMN].max()}")
print(f"\nQuarterly distribution:")
print(df["quarter"].value_counts().sort_index())
print(f"\nSample doc (first 300 chars):")
print(docs[0][:300])

Documents ready: 458
Time range: 2019-01-10 20:55:47 to 2025-09-11 07:00:05

Quarterly distribution:
quarter
2019-01-01      2
2019-04-01      1
2019-07-01      1
2020-01-01      2
2021-07-01      1
2021-10-01      1
2022-04-01      1
2022-07-01      3
2022-10-01      2
2023-01-01     12
2023-04-01     16
2023-07-01     15
2023-10-01     16
2024-01-01     20
2024-04-01     22
2024-07-01     20
2024-10-01     23
2025-01-01     60
2025-04-01     89
2025-07-01    151
Name: count, dtype: int64

Sample doc (first 300 chars):
Accidentally screwed over coworkers because of ChatGPT, what do I do? Hi. During a meeting like two weeks ago, my manager brought up the topic of AI in the workplace. I said that while I found it a great tool, I felt that we should be careful when using it while talking with clients (we are a consul


## 3. Configure BERTopic with Noise Reduction

Key strategies to reduce noise words ("sisters", "that", "like", etc.):
1. **CountVectorizer** with `stop_words="english"` removes common English stopwords
2. **Custom stopwords** for domain-specific noise (e.g., Reddit jargon)
3. **KeyBERTInspired** representation model produces more coherent, less noisy topic labels
4. **min_df** filters out very rare terms; **ngram_range** captures multi-word phrases

In [7]:
# Custom stopwords: add domain-specific noise words here
# These are words that appear frequently but don't help distinguish topics
custom_stopwords = [
    # Reddit-specific noise
    "like", "just", "got", "get", "going", "would", "could", "really",
    "also", "know", "think", "want", "even", "still", "much", "thing",
    "things", "way", "make", "made", "said", "one", "people", "time",
    "don", "didn", "doesn", "isn", "wasn", "won", "wouldn", "couldn",
    "shouldn", "hasn", "hadn", "aren", "weren", "ll", "ve",
    # Conversational noise commonly seen in Reddit posts/comments
    "yeah", "okay", "lol", "lmao", "edit", "update", "deleted",
    "comment", "comments", "post", "posted", "thread", "subreddit",
    "reddit", "op", "username",
    # Relationship/story noise often appearing in work-related posts
    "sister", "sisters", "brother", "husband", "wife", "friend",
    "mom", "dad", "family",
]

# Merge with sklearn's English stopwords
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
all_stopwords = list(ENGLISH_STOP_WORDS.union(custom_stopwords))

print(f"Total stopwords: {len(all_stopwords)}")

Total stopwords: 373


In [8]:
# --- Sub-models ---
# Each sub-model below controls a different stage of the BERTopic pipeline.
# Modify parameters to change how topics are discovered and represented.

# ---- Embedding model ----
# Converts documents into dense vector representations.
# Try "all-mpnet-base-v2" for higher quality (slower) or "paraphrase-MiniLM-L3-v2" for speed.
# Full list: https://www.sbert.net/docs/sentence_transformer/pretrained_models.html
embedding_model = SentenceTransformer("all-roberta-large-v1")

# ---- UMAP: dimensionality reduction ----
# Reduces high-dimensional embeddings before clustering.
umap_model = UMAP(
    n_neighbors=15,      # Controls local vs global structure. Lower (5-10) = finer local detail;
                         # higher (30-50) = broader global structure. Try 10 or 30.
    n_components=5,      # Target dimensions. 5 is a good default. Try 10 if you have many docs.
    min_dist=0.0,        # How tightly points cluster. 0.0 = tight clusters (best for topic modeling).
    metric="cosine",     # Distance metric. "cosine" works well for text embeddings.
    random_state=42,     # Set for reproducibility. Change or remove for different runs.
)

# ---- HDBSCAN: clustering ----
# Groups reduced embeddings into topic clusters.
hdbscan_model = HDBSCAN(
    min_cluster_size=15,          # *** KEY PARAMETER *** Minimum docs to form a topic.
                                  # LOWER (5-10) = more topics, smaller clusters.
                                  # HIGHER (30-50) = fewer topics, broader clusters.
                                  # Start with 10-15 and adjust based on results.
    min_samples=None,             # Controls how conservative clustering is. None = same as
                                  # min_cluster_size. Lower values (1-5) = less noise/outliers.
    metric="euclidean",           # Distance metric for clustering.
    cluster_selection_method="eom",  # "eom" = variable cluster sizes (recommended).
                                     # "leaf" = more uniform, smaller clusters.
    prediction_data=True,         # Required for soft clustering / probability output.
)

# ---- CountVectorizer: tokenization & stopwords ----
# Controls how documents are tokenized for topic representation (c-TF-IDF).
# This does NOT affect clustering -- only the words shown in topic labels.
vectorizer_model = CountVectorizer(
    stop_words=all_stopwords,  # Words to exclude. Add noisy words to custom_stopwords above.
    min_df=2,                  # Ignore terms appearing in fewer than N documents.
                               # Increase (e.g., 5 or 10) to remove rare/noisy words.
    max_df=0.95,               # Ignore terms appearing in more than 95% of documents.
                               # Lower (e.g., 0.8) to remove overly common words.
    ngram_range=(1, 2),        # (1,1) = single words only. (1,2) = unigrams + bigrams.
                               # (1,3) = up to trigrams (e.g., "machine learning engineer").
)

# ---- Representation model: KeyBERTInspired ----
# Refines topic labels using embedding similarity (reduces noise words significantly).
representation_model = KeyBERTInspired(
    top_n_words=10,            # Number of words per topic in final representation.
    nr_repr_docs=5,            # Number of representative docs used per topic. Increase for
                               # more diverse representations.
    nr_samples=500,            # Candidate docs sampled per cluster before selecting repr docs.
    nr_candidate_words=100,    # Candidate words considered before selecting top_n_words.
    random_state=42,           # Set for reproducibility.
)
# Alternative: use MaximalMarginalRelevance for more diverse topic words:
#   from bertopic.representation import MaximalMarginalRelevance
#   representation_model = MaximalMarginalRelevance(diversity=0.3)
#   diversity: 0.0 = most similar words, 1.0 = most diverse words. Try 0.2-0.5.

print("Sub-models configured.")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: sentence-transformers/all-roberta-large-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sub-models configured.


## 4. Fit BERTopic Model

In [9]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,
    top_n_words=10,        # Words shown per topic. Increase to see more words per topic.
    nr_topics=None,        # Set to an integer (e.g., 10) to force that many topics,
                           # or "auto" to automatically merge similar topics.
                           # None = let HDBSCAN decide the number of topics.
    verbose=True,
)

topics, probs = topic_model.fit_transform(docs)

print(f"\nNumber of topics found: {len(set(topics)) - (1 if -1 in topics else 0)}")
print(f"Outlier documents (topic -1): {topics.count(-1)}/{len(topics)}")

2026-03-06 03:27:17,839 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

2026-03-06 03:27:20,413 - BERTopic - Embedding - Completed ✓
2026-03-06 03:27:20,422 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-06 03:27:24,463 - BERTopic - Dimensionality - Completed ✓
2026-03-06 03:27:24,464 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-06 03:27:24,471 - BERTopic - Cluster - Completed ✓
2026-03-06 03:27:24,473 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-06 03:27:25,288 - BERTopic - Representation - Completed ✓



Number of topics found: 2
Outlier documents (topic -1): 11/458


In [10]:
# View discovered topics
topic_info = topic_model.get_topic_info()
print("Discovered topics:")
topic_info

Discovered topics:


,Topic,Count,Name,Representation,Representative_Docs
0,-1,11,-1_ai careers_smith ai_hr calls_employer custo...,"[ai careers, smith ai, hr calls, employer cust...",[How demonstrating your willingness to learn c...
1,0,230,0_fired_unemployment_situation_job market,"[fired, unemployment, situation, job market, w...","[I work from home and barely do anything, will..."
2,1,217,1_use ai_layoffs_machine_openai,"[use ai, layoffs, machine, openai, automate, h...",[I run an AI automation agency (AAA). My hones...


In [11]:
# Show top words for each topic
for topic_id in topic_info[topic_info["Topic"] != -1]["Topic"]:
    words = [w for w, _ in topic_model.get_topic(topic_id)]
    print(f"Topic {topic_id}: {', '.join(words[:8])}")

Topic 0: fired, unemployment, situation, job market, workers, quit, hired, employment
Topic 1: use ai, layoffs, machine, openai, automate, humans, adobe, corporate


## 5. Static Visualizations

In [13]:
# Topic word barchart
# top_n_topics: how many topics to show (e.g., 5, 10, 20). Set to None for all.
# n_words: how many words per topic bar.
fig = topic_model.visualize_barchart(top_n_topics=10, n_words=8)
fig.show()
fig.write_html(f"{OUTPUT_DIR}/cluster6_barchart.html")
print(f"Saved: {OUTPUT_DIR}/cluster6_barchart.html")

Saved: outputs/cluster6_barchart.html


In [14]:
# Intertopic distance map (2D projection of topic similarity)
# Requires at least 3 non-outlier topics for UMAP reduction to work.
num_topics = len([t for t in set(topics) if t != -1])


if num_topics >= 3:
    fig = topic_model.visualize_topics()
    fig.show()
    fig.write_html(f"{OUTPUT_DIR}/cluster6_intertopic_map.html")
    print(f"Saved: {OUTPUT_DIR}/cluster6_intertopic_map.html")
else:
    print(f"Skipped: only {num_topics} topic(s) found (need >= 3 for intertopic map).")
    print("Try lowering min_cluster_size or adding more data.")


Skipped: only 2 topic(s) found (need >= 3 for intertopic map).
Try lowering min_cluster_size or adding more data.


In [15]:
# Topic hierarchy (dendrogram of topic similarity)
# Requires at least 2 non-outlier topics.
if num_topics >= 2:
    fig = topic_model.visualize_hierarchy()
    fig.show()
    fig.write_html(f"{OUTPUT_DIR}/cluster6_hierarchy.html")
    print(f"Saved: {OUTPUT_DIR}/cluster6_hierarchy.html")
else:
    print(f"Skipped: only {num_topics} topic(s) found (need >= 2 for hierarchy).")
    print("Try lowering min_cluster_size or adding more data.")

Saved: outputs/cluster6_hierarchy.html


## 6. Dynamic Topic Modeling (Topics Over Time)

Calculates how topic representations evolve across **quarterly** timestamps.

The timestamps were pre-binned to quarters in Section 2, so no additional binning is needed.

In [16]:
# Verify quarterly timestamp distribution
unique_quarters = sorted(df["quarter"].unique())
print(f"Unique quarters ({len(unique_quarters)}):")
for q in unique_quarters:
    count = (df["quarter"] == q).sum()
    print(f"  {q.strftime('%Y-Q')}{(q.month - 1) // 3 + 1}: {count} docs")

Unique quarters (20):
  2019-Q1: 2 docs
  2019-Q2: 1 docs
  2019-Q3: 1 docs
  2020-Q1: 2 docs
  2021-Q3: 1 docs
  2021-Q4: 1 docs
  2022-Q2: 1 docs
  2022-Q3: 3 docs
  2022-Q4: 2 docs
  2023-Q1: 12 docs
  2023-Q2: 16 docs
  2023-Q3: 15 docs
  2023-Q4: 16 docs
  2024-Q1: 20 docs
  2024-Q2: 22 docs
  2024-Q3: 20 docs
  2024-Q4: 23 docs
  2025-Q1: 60 docs
  2025-Q2: 89 docs
  2025-Q3: 151 docs


In [17]:
topics_over_time = topic_model.topics_over_time(
    docs,
    timestamps,              # Already quarterly-binned timestamps from Section 2.
    # nr_bins=None,          # Not needed since timestamps are already quarterly.
                             # Uncomment and set (e.g., 10) if you want further binning.
    evolution_tuning=True,   # Average each timestep's representation with the previous timestep.
                             # Set False to see raw per-quarter representations.
    global_tuning=True,      # Average each timestep's representation with the global representation.
                             # Set False to see purely local per-quarter representations.
)

print(f"Topics over time shape: {topics_over_time.shape}")
topics_over_time.head(10)

20it [00:03,  5.27it/s]

Topics over time shape: (38, 4)


,Topic,Words,Frequency,Timestamp
0,1,"sales marketing, linkedin marketing, product m...",2,2019-01-01
1,1,"data science, data scientist, machine learning...",1,2019-04-01
2,1,"recruitment, job seeking, recruitment process,...",1,2019-07-01
3,0,"vacation days, work life, vacation, vacations,...",1,2020-01-01
4,1,"ai chatbot, customer service, chatbots, chatbo...",1,2020-01-01
5,1,"fintech, stocks, investors, sell, startup",1,2021-07-01
6,1,"eeoc, use ai, engineering, openai, workplace",1,2021-10-01
7,0,"fired weeks, weeks notice, working amazon, ama...",1,2022-04-01
8,0,"resumes, attorney, administrative, attorneys, ...",2,2022-07-01
9,1,"certified, certificate, diploma, 100 free, certs",1,2022-07-01


In [18]:
# Visualize topics over time
# top_n_topics: how many topics to plot (e.g., 5, 10). Set to None for all.
# topics: pass a list of specific topic IDs to plot, e.g., topics=[0, 1, 3].
# normalize_frequency: set True to normalize each topic's frequency (useful for comparison).
fig = topic_model.visualize_topics_over_time(
    topics_over_time,
    top_n_topics=10,
)
fig.show()
fig.write_html(f"{OUTPUT_DIR}/cluster6_topics_over_time.html")
print(f"Saved: {OUTPUT_DIR}/cluster6_topics_over_time.html")

Saved: outputs/cluster6_topics_over_time.html


## 7. Topics Per Subreddit (Optional Analysis)

In [19]:
# Analyze which topics appear in which subreddits
# top_n_topics: how many topics to show (e.g., 5, 10). Set to None for all.
subreddits = df["subreddit"].tolist()
topics_per_class = topic_model.topics_per_class(docs, classes=subreddits)

fig = topic_model.visualize_topics_per_class(
    topics_per_class,
    top_n_topics=10,
)
fig.show()
fig.write_html(f"{OUTPUT_DIR}/cluster6_topics_per_subreddit.html")
print(f"Saved: {OUTPUT_DIR}/cluster6_topics_per_subreddit.html")

91it [00:10,  9.06it/s]


Saved: outputs/cluster6_topics_per_subreddit.html


## 8. Explore Specific Topics

In [20]:
# Find representative documents for each topic
for topic_id in sorted(set(topics)):
    if topic_id == -1:
        continue
    repr_docs = topic_model.get_representative_docs(topic_id)
    words = [w for w, _ in topic_model.get_topic(topic_id)]
    print(f"\n{'='*80}")
    print(f"Topic {topic_id}: {', '.join(words[:6])}")
    print(f"{'='*80}")
    for i, doc in enumerate(repr_docs[:2]):
        print(f"  Doc {i+1}: {doc[:200]}...")


Topic 0: fired, unemployment, situation, job market, workers, quit
  Doc 1: I work from home and barely do anything, will I get fired for this? A year ago, I landed a fully remote job as a project coordinator for a MNC. The interview process was tough - behavioral rounds, a c...
  Doc 2: What obsolete "Zoomer" advice will we give our children? There are many complaints of Boomer / Gen-X parents giving terrible advice about the job market. It's become a cliche. "Walk in and demand to s...

Topic 1: use ai, layoffs, machine, openai, automate, humans
  Doc 1: I run an AI automation agency (AAA). My honest overview and review of this new business model I started an AI tools directory in February, and then branched off that to start an AI automation agency (...
  Doc 2: Tech Layoffs: The Harsh Reality & What You Need to Know After speaking with friends at Microsoft, Meta, and Amazon across London, Bangalore, and Seattle, here are the hard truths about the current job...


In [12]:
import pandas as pd

rows = []

# Find representative documents for each topic
for topic_id in sorted(set(topics)):
    if topic_id == -1:
        continue
        
    repr_docs = topic_model.get_representative_docs(topic_id)
    words = [w for w, _ in topic_model.get_topic(topic_id)]

    topic_words = ", ".join(words[:6])

    for i, doc in enumerate(repr_docs[:2]):
        rows.append({
            "topic_id": topic_id,
            "topic_words": topic_words,
            "doc_id": i+1,
            "document": doc
        })

df = pd.DataFrame(rows)
df.to_csv("topic_representative_docs.csv", index=False)


CSV 已导出: topic_representative_docs.csv


In [14]:
fig.hist(bertopic_prob, bins=50)
fig.title("Topic Probability Distribution")
fig.xlabel("Probability")
fig.show()

NameError: name 'fig' is not defined

## 9. Save Results

In [21]:
# Save topic assignments back to the dataframe
df["bertopic_topic"] = topics
df["bertopic_prob"] = [p.max() if hasattr(p, 'max') else p for p in probs]

# Save to CSV
output_path = f"{OUTPUT_DIR}/cluster6_topic_results.csv"
df.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")

# Save topics over time
tot_path = f"{OUTPUT_DIR}/cluster6_topics_over_time.csv"
topics_over_time.to_csv(tot_path, index=False)
print(f"Topics over time saved to {tot_path}")

Results saved to outputs/cluster6_topic_results.csv
Topics over time saved to outputs/cluster6_topics_over_time.csv


In [22]:
# Save the model (optional - uncomment to save)
topic_model.save(f"{OUTPUT_DIR}/cluster6_bertopic_model", serialization="safetensors", save_ctfidf=True)

---

## Tips for Better Results

**If you still see noise words in topics:**
1. Add them to `custom_stopwords` in Section 3 and re-run from there
2. Increase `min_df` in the CountVectorizer (e.g., `min_df=5`)
3. Try `MaximalMarginalRelevance` as an alternative representation model:
   ```python
   from bertopic.representation import MaximalMarginalRelevance
   representation_model = MaximalMarginalRelevance(diversity=0.3)
   ```

**If too many documents are outliers (topic -1):**
1. Lower `min_cluster_size` in HDBSCAN (e.g., `min_cluster_size=10`)
2. Use `topic_model.reduce_outliers(docs, topics)` to reassign outliers

**If too few / too many topics:**
1. Set `nr_topics` in BERTopic constructor to control the number (e.g., `nr_topics=10`)
2. Or use `nr_topics="auto"` for automatic reduction
3. Adjust `min_cluster_size` — larger values = fewer, broader topics